# Revised answer-first synthetic BioASQ factoid QA pilot (v2)

This notebook runs the revised **20-source quality pilot** with GPT-4.1 Mini as both generator and blinded verifier. It does not reuse or overwrite the completed v1 model-comparison run.

The v2 safeguards are:

- A source may be declared ineligible unless it supports **two distinct single-answer biomedical relations**.
- List answers, document-metadata questions, answer-leaking/tautological questions, and near-duplicate relations are rejected locally.
- Answers remain exact continuous spans; punctuation and hyphens are preserved.
- Finalization accepts an exact verifier match or a longer verifier clause that wholly contains the intended answer. Reverse containment is rejected because it may omit a required qualifier.
- The real 160-question dev split and protected test sources remain excluded.

Start with `SOURCE_LIMIT = 20`. Review the accepted and rejected records before changing it to `0` for all 250 prepared sources.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

PROJECT_ROOT = Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
PYTHON = Path('/home/dinh-tuan/miniconda3/envs/bioasq/bin/python')
SCRIPT = PROJECT_ROOT / 'cse_dpo/build_synthetic_factoid_qa_pilot.py'
OUTPUT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/synthetic_factoid_qa/gpt41mini_answer_first_250x2_v2'
GENERATOR_MODEL = 'gpt-4.1-mini-2025-04-14'
VERIFIER_MODEL = 'gpt-4.1-mini-2025-04-14'
SOURCE_LIMIT = 20

def run_phase(phase, *, dry_run=False):
    cmd = [
        str(PYTHON), str(SCRIPT),
        '--phase', phase,
        '--output-root', str(OUTPUT_ROOT),
        '--generator-model', GENERATOR_MODEL,
        '--verifier-model', VERIFIER_MODEL,
        '--source-count', '250',
        '--validation-source-count', '50',
        '--retrieval-candidate-count', '100',
        '--source-limit', str(SOURCE_LIMIT),
        '--progress-every', '5',
    ]
    if dry_run:
        cmd.append('--dry-run')
    print(' '.join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

print('Output root:', OUTPUT_ROOT)


## 1. Inspect the prepared source manifest

`prepare` is local and makes no API calls. The v2 manifest has already been prepared, but rerunning it is deterministic.


In [ ]:
run_phase('prepare')


In [ ]:
prepare_summary = json.loads((OUTPUT_ROOT / 'prepare_summary.json').read_text())
prepare_summary


In [ ]:
manifest = pd.read_json(OUTPUT_ROOT / 'source_manifest.jsonl', lines=True)
print(manifest.head(SOURCE_LIMIT).groupby(['split', 'evidence_arm']).size())
manifest[['synthetic_source_id', 'split', 'evidence_arm']].head(SOURCE_LIMIT)


## 2. Preview the revised generation prompt

This prints one prompt and makes no API calls. Confirm that the prompt permits `eligible=false` and forbids list, metadata, and tautological questions.


In [ ]:
run_phase('generate', dry_run=True)


## 3. Generate the 20-source pilot

This makes at most 20 generator calls. Ineligible sources produce zero questions and remain part of the pilot denominator.


In [ ]:
run_phase('generate')


In [ ]:
generation_summary = json.loads((OUTPUT_ROOT / 'generation_summary.json').read_text())
generation_summary


In [ ]:
generated = pd.read_json(OUTPUT_ROOT / 'generated_sources.jsonl', lines=True)
generated[['synthetic_source_id', 'evidence_arm', 'generation_eligible',
           'generation_eligibility_basis']].head(SOURCE_LIMIT)


In [ ]:
generated_items = pd.DataFrame([
    {
        'source_id': source['synthetic_source_id'],
        'arm': source['evidence_arm'],
        'pmid': source['target']['pubmed_id'],
        'question_index': index,
        **item,
    }
    for source in generated.to_dict('records')
    for index, item in enumerate(source.get('generated_items', []), 1)
])
generated_items


## 4. Preview and run blinded verification

The verifier sees the final clean or distractor evidence pack, but never sees the intended answers. Explicitly ineligible sources are skipped.


In [ ]:
run_phase('verify', dry_run=True)


In [ ]:
run_phase('verify')


In [ ]:
verification_summary = json.loads((OUTPUT_ROOT / 'verification_summary.json').read_text())
verification_summary


## 5. Finalize locally

Finalization makes no API calls. It reapplies all generation checks, requires unique explicit support from the target resource, and uses the directional containment rule for verifier agreement.


In [ ]:
run_phase('finalize')


In [ ]:
final_summary = json.loads((OUTPUT_ROOT / 'final_summary.json').read_text())
final_summary


In [ ]:
accepted = pd.read_json(OUTPUT_ROOT / 'synthetic_accepted_all.jsonl', lines=True)
accepted[['synthetic_question_id', 'evidence_arm', 'question', 'answer',
          'answer_type', 'verification_match_type']]


## 6. Inspect every rejected or ambiguous record

Pay particular attention to `generation_validation_error`, `verification_match_type`, wrong-resource selections, and distractor-only failures.


In [ ]:
review = pd.read_csv(OUTPUT_ROOT / 'synthetic_review.csv')
review


## 7. Scale only after the 20-source audit

If the questions are specific, non-tautological, and unambiguous under distractors, set `SOURCE_LIMIT = 0` and run `generate`, `verify`, and `finalize` again. Cached pilot calls will be reused. The final training files are:

- `synthetic_train_prepared.json`
- `synthetic_validation_prepared.json`

Do not mix these synthetic validation examples into the real 160-question dev split.


In [ ]:
# After setting SOURCE_LIMIT = 0 in the settings cell, run:
# run_phase('generate')
# run_phase('verify')
# run_phase('finalize')
